In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


import warnings
warnings.filterwarnings("ignore")

 # 第一步：数据处理与中文化处理

 - 获取数据并将数据中文化方便阅读

In [ ]:
df = pd.read_csv('多种高发癌症数据数据集.csv')
# 列名转中文
col_map = {
    'Study ID': '研究编号',
    'Patient ID': '患者编号',
    'Sample ID': '样本编号',
    'Tumor Stage': '肿瘤分期',
    'Cancer Type': '癌症类型',
    'Cancer Type Detailed': '癌症类型详细',
    'Sample Coverage': '样本覆盖度',
    'Diagnosis Age': '诊断年龄',
    'DNA Input': 'DNA输入量',
    'Mutation Count': '突变数量',
    'Number of Samples Per Patient': '每位患者样本数',
    'Sample Type': '样本类型',
    'Sex': '性别',
    'Smoke Status': '吸烟状况',
    'Specimen Preservation Type': '标本保存类型',
    'Specimen Type': '标本类型',
    'TMB (nonsynonymous)': 'TMB（非同义）',
    'Treatment': '治疗',
    'Tumor Purity': '肿瘤纯度'
}
df = df.rename(columns=col_map)
# 分类值转中文
value_maps = {
    '肿瘤分期': {'I':'I期','II':'II期','III':'III期','IV':'IV期','I-II':'I-II期','III-IV':'III-IV期','Unknown':'未知','0':'0期'},
    '样本类型': {'Primary':'原发','Metastasis':'转移','Recurrent':'复发'},
    '性别': {'Female':'女性','Male':'男性'},
    '吸烟状况': {'Unknown':'未知','Nonsmoker':'不吸烟','Smoker':'吸烟'},
    '标本保存类型': {'FFPE':'福尔马林固定石蜡包埋'},
    '标本类型': {'Surgery':'手术','Biopsy/Paracentesis':'活检/穿刺'},
    '治疗': {
        'Treatment-naive':'未治疗',
        'Chemotherapy':'化疗',
        'Radiation_Therapy':'放疗',
        'Targeted_Therapy':'靶向治疗',
        'Immunotherapy':'免疫治疗',
        'Endocrinotherapy':'内分泌治疗',
        'Other_Treatments':'其他治疗',
        'Unknown':'未知'
    },
    '癌症类型': {
        'Colorectal Carcinoma':'结直肠癌',
        'Liver Hepatocellular Carcinoma':'肝细胞癌',
        'Soft Tissue Sarcoma':'软组织肉瘤',
        'Melanoma':'黑色素瘤',
        'Carcinoma of Uterine Cervix':'子宫颈癌',
        'Urothelial Carcinoma':'尿路上皮癌',
        'Thymic Tumor':'胸腺肿瘤',
        'Kidney Renal Cell Carcinoma':'肾细胞癌',
        'Ovarian Carcinoma':'卵巢癌',
        'Gastric Cancer':'胃癌',
        'Pancreatic Cancer':'胰腺癌',
        'Breast Carcinoma':'乳腺癌',
        'Extrahepatic Cholangiocarcinoma':'肝外胆管癌',
        'Bone Sarcoma':'骨肉瘤',
        'Esophageal Carcinoma':'食管癌',
        'Small Cell Lung Cancer':'小细胞肺癌',
        'Non Small Cell Lung Cancer':'非小细胞肺癌',
        'Cancer of Unknown Primary':'原发灶不明癌',
        'Intrahepatic Cholangiocarcinoma':'肝内胆管癌',
        'Uterine Corpus Endometrial Carcinoma':'子宫体子宫内膜癌',
        'Head and Neck Carcinoma':'头颈癌',
        'Gallbladder Carcinoma':'胆囊癌',
        'Gastrointestinal Neuroendocrine Tumor':'胃肠神经内分泌肿瘤',
        'Small Bowel Carcinoma':'小肠癌',
        'Thyroid Carcinoma':'甲状腺癌'
    },
    '癌症类型详细': {
        'Colorectal Adenocarcinoma': '结直肠腺癌',
        'Hepatocellular Carcinoma': '肝细胞癌',
        'Gastrointestinal Stromal Tumor': '胃肠间质瘤',
        'Cutaneous Melanoma': '皮肤黑色素瘤',
        'Endometrial Adenocarcinoma': '子宫内膜腺癌',
        'Bladder Urothelial Carcinoma': '膀胱尿路上皮癌',
        'Thymoma': '胸腺瘤',
        'Renal Clear Cell Carcinoma': '肾透明细胞癌',
        'High-Grade Serous Ovarian Cancer': '高级别浆液性卵巢癌',
        'Gastric Adenocarcinoma': '胃腺癌',
        'Pancreatic Adenocarcinoma': '胰腺腺癌',
        'Breast Invasive Carcinoma': '浸润性乳腺癌',
        'Esophageal Squamous Cell Carcinoma': '食管鳞状细胞癌',
        'Lung Adenocarcinoma': '肺腺癌',
        'Lung Squamous Cell Carcinoma': '肺鳞状细胞癌',
        'Clear Cell Sarcoma': '透明细胞肉瘤',
        'Cancer of Unknown Primary': '原发灶不明癌',
        'Small Cell Lung Cancer': '小细胞肺癌',
        'Nasopharyngeal Carcinoma': '鼻咽癌',
        'Leiomyosarcoma': '平滑肌肉瘤'      
    }
}

# 循环应用字典映射
for col, mapping in value_maps.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)
# 治疗列特殊处理（可能包含多个治疗，用分号分隔）
def translate_treatment(x):
    if pd.isna(x):
        return x
    parts = str(x).split(';')
    treat_map = value_maps['治疗']
    return '；'.join(treat_map.get(p, p) for p in parts)
if '治疗' in df.columns:
    df['治疗'] = df['治疗'].apply(translate_treatment)
# 导出中文 CSV
df.to_csv('多种高发癌症数据数据集_中文.csv', index=False, encoding='utf-8-sig')
print('已生成：多种高发癌症数据数据集_中文.csv')
df.head(1000)

# 第二步：数据预处理
 - 去除无用的编号

In [ ]:
df.drop(columns=["研究编号", "患者编号", "样本编号"], inplace=True)
df.head(1000)

# 第三步：数据探索性分析

In [ ]:
# 查看缺失值情况
df.isnull().sum()

In [ ]:
# 查看数据的整体结构、每列的数据类型和缺失情况
df.info()

In [ ]:
# 查看数值型列的统计信息（均值、方差、最小最大值等）
df.describe()

In [ ]:
ProfileReport(df)#生成完整的数据探索性分析（EDA）报告

In [ ]:
# 解决中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows系统用黑体
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # 如果你是Mac电脑，请用这行替换上一行
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

 - 1.不同癌症类型下原发/转移/复发样本数量分布

In [ ]:
plt.figure(figsize=(14, 6)) # 增加画布宽度防止横坐标挤在一起
sns.countplot(x="癌症类型", hue="样本类型", data=df, palette="rainbow")
plt.title('不同癌症类型下原发/转移/复发样本数量对比') # 加上标题
plt.xticks(rotation=90) 
plt.show()

从上图可以看出，数据集中样本类型呈现极度不平衡状态。绝大多数癌症类型以原发灶（Primary）为主，转移灶（Metastasis）和复发灶（Recurrent）样本量极少。这提示在后续建模时，模型极易陷入“多数类陷阱”，必须使用 stratify 划分和 class_weight 平衡。

 - 2.诊断年龄与样本类型的潜在关系

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="样本类型", y="诊断年龄", data=df, palette="rainbow")
plt.title('不同样本类型的诊断年龄分布')
plt.show()

 从箱线图可以看出，复发灶（Recurrent）患者的年龄中位数明显高于原发和转移灶。这提示高龄可能是癌症复发的一个重要风险因素，该特征需重点关注。

 - 3.突变数量、TMB与样本类型的关联分析

In [ ]:
plt.figure(figsize=(12, 5))
# 突变数量
plt.subplot(1, 2, 1) # 1行2列，第1个图
sns.boxplot(x="样本类型", y="突变数量", data=df, palette="rainbow")
plt.title('不同样本类型的突变数量分布')

# TMB
plt.subplot(1, 2, 2) # 1行2列，第2个图
sns.boxplot(x="样本类型", y="TMB（非同义）", data=df, palette="rainbow")
plt.title('不同样本类型的TMB分布')

plt.tight_layout() # 自动调整子图间距
plt.show()

 - 4.肿瘤纯度与样本类型的显著差异

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="样本类型", y="肿瘤纯度", data=df, palette="rainbow")
plt.title('不同样本类型的肿瘤纯度分布')
plt.show()

 原发性样本的突变数量和TMB波动范围极大，存在许多超突变极值，而转移和复发灶数值相对集中。这些真实的生物学极值在后期的数据预处理中绝不能盲目作为“错误数据”剔除，它们极有可能是模型区分类型的关键特征。

# 第4步：建模分析

- 数据去重与异常值标记

### 1. 数据清洗：去重与异常值处理
* **去重**：检查是否有重复样本，确保数据唯一性。
* **异常值处理思路**：医疗数据中，“肿瘤纯度”理论上应在0-100之间。但我没有直接删除超出范围的值，而是采用**“先标记，不盲目删除”**的原则。因为医学数据中极高的值可能是真实的超突变表型，直接删除会丢失关键的生物学信息。

In [ ]:
print("去重前:", df.shape)
df = df.drop_duplicates()
print("去重后:", df.shape)
# 去掉重复数据
df['purity_abnormal'] = (df['肿瘤纯度'] < 0) | (df['肿瘤纯度'] > 100)
# 检查异常值（先标记，不盲目删除）

 - 特征工程

### 2. 特征工程：衍生新变量
* **年龄分箱**：将连续年龄切分为年龄段（如<30, 30-50等），为了让模型更容易捕捉年龄阶段的非线性规律。
* **TMB高低分组**：以中位数为界进行二分类。
* **注意点**：这个步骤必须在划分训练集和测试集之前完成。作为实验练习，这是为了之后能直观地看到不同年龄段/不同TMB水平对预测的影响。

In [ ]:
df['年龄分组'] = pd.cut(df['诊断年龄'], bins=[0, 30, 50, 70, 120], labels=['<30', '30-50', '50-70', '>70'])
tmb_median = df['TMB（非同义）'].median()
df['TMB_等级'] = np.where(df['TMB（非同义）'] > tmb_median, '高TMB', '低TMB')

 - 数据划分与构建预处理管道

### 3. 数据划分与预处理管道
* **stratify=y**：因为数据极度不平衡（原发灶极多，复发灶极少），必须使用分层抽样，保证测试集里各类别的比例跟整体一致，否则模型测试结果会失真。
* **Pipeline管道构建**：
    * 数值型特征（中位数填补缺失值 + 标准化）：消除年龄和测序深度等不同量纲的差异。
    * 类别型特征（众数填补 + 独热编码）：防止模型把分类误认为数字大小。
* **数据防泄露**：所有填充和标准化都放进管道里，确保只在训练集上计算规则，绝不会把测试集的信息泄露给训练过程。

In [ ]:
# 划分特征与标签
model_data = df.dropna(subset=['样本类型'])
model_data = model_data[model_data['样本类型'].isin(['原发', '转移', '复发'])]
X = model_data.drop(columns=['样本类型'])
y = model_data['样本类型']
#划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#定义数值型特征和类别型特征
numerical_features = ['诊断年龄', '样本覆盖度', 'DNA输入量', '突变数量', 'TMB（非同义）', '肿瘤纯度']
categorical_features = ['肿瘤分期', '癌症类型', '性别', '吸烟状况', '标本类型', '治疗', '年龄分组', 'TMB_等级']
#构建基础预处理管道
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numerical_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ])
print("前置数据准备与基础管道构建完成！")

 - 构建基线模型并评估

### 4. 随机森林基线模型
* **算法选择**：随机森林，能处理混合特征、非线性关系，且自带特征重要性。
* **class_weight='balanced'**：因为类别不平衡，加入类别权重平衡，让少数类（转移/复发）对模型损失有更大的影响力。
* **评估指标**：不能只看准确率，必须看分类报告（F1-score）和混淆矩阵，重点关注少数类（复发灶）是否被识别。

In [ ]:
# 构建基础随机森林模型
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 训练与预测
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print("【基线模型】测试集准确率:", accuracy_score(y_test, y_pred_rf))
print("【基线模型】分类报告:\n", classification_report(y_test, y_pred_rf))

# 混淆矩阵可视化
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues',
            xticklabels=model_rf.classes_, yticklabels=model_rf.classes_)
plt.title('基线模型 - 混淆矩阵')
plt.xlabel('预测类别')
plt.ylabel('真实类别')
plt.show()

 - 网格搜索调优

### 5. 模型调优（网格搜索）
* **自动调参**：使用GridSearchCV，定义树的深度、数量等参数网格。
* **关键改动**：把评分标准 `scoring` 从默认的 `accuracy` 改成了 `f1_macro`。因为我们的数据极度不平衡，准确率容易被多数类（原发灶）骗高。必须用对少数类更友好的F1-macro来引导参数搜索，这也是结合业务实际对AI建议的修正。

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10]
}
#网格搜索
grid_search = GridSearchCV(model_rf, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("最佳参数组合:", grid_search.best_params_)
best_model_rf = grid_search.best_estimator_
#调优后评估
y_pred_best_rf = best_model_rf.predict(X_test)
print("【调优后】测试集准确率:", accuracy_score(y_test, y_pred_best_rf))
print("【调优后】分类报告:\n", classification_report(y_test, y_pred_best_rf))

 - 特征重要性与单样本预测

### 6. 特征重要性分析
* **特征重要性**：提取模型判断的依据。我们发现“肿瘤分期IV期”排第一。结合医学常识，IV期本身就意味着已经发生转移，这提示模型可能存在“数据泄露”。

In [ ]:
# 提取特征重要性
feature_names = best_model_rf.named_steps['preprocessor'].get_feature_names_out()
importances = best_model_rf.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({'特征': feature_names, '重要性': importances})
feature_importance_df['特征'] = feature_importance_df['特征'].str.replace('num__', '').str.replace('cat__', '')
feature_importance_df = feature_importance_df.sort_values(by='重要性', ascending=False)

# 可视化
plt.figure(figsize=(10, 6))
sns.barplot(x='重要性', y='特征', data=feature_importance_df.head(15), palette='viridis')
plt.title('特征重要性 Top 15')
plt.show()

### 7. 模型强化与优化

**模型强化：引入 SMOTE 与 XGBoost 结合样本权重。**
针对基线模型在“复发/转移”类别上严重偏向多数类的问题，我引入了 SMOTE 过采样技术平衡训练集，并采用 XGBoost 算法替代随机森林。同时，由于 XGBoost 严格要求目标标签 y 为数值型（0, 1, 2），而之前用的是中文文本（原发、转移、复发），因此先用 `LabelEncoder` 进行标签数值化映射，完成后再次反转回中文以便查看预测结果。
**核心改进策略**：
在之前的尝试中，单纯依赖 SMOTE 或单纯依赖样本权重，都会出现各种问题。因此本次使用了“**双重强化**”策略：先进行 SMOTE 过采样生成合成样本，再基于过采样后的数据计算样本权重，完美解决了维度不匹配的问题，大幅提升了模型对少数类（复发/转移）的识别能力。

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import pickle  # 必须导入 pickle 用于保存

#标签数值化
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
print("标签编码对应关系：", dict(zip(le.classes_, le.transform(le.classes_))))

#先在训练集上拟合预处理管道，并转换训练集和测试集
#注意：只对训练集做 fit_transform，测试集只能 transform，严防数据泄露
# 注意：这里 preprocessor 变量必须是你的预处理器对象（比如 ColumnTransformer 或 StandardScaler），千万不要覆盖它！
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# 对训练集进行 SMOTE 过采样
smote = SMOTE(random_state=42, sampling_strategy='not majority')
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_prep, y_train_encoded)

#计算过采样后的样本权重
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_resampled)

#训练 XGBoost
model_advanced = xgb.XGBClassifier(
    random_state=42, 
    eval_metric='mlogloss',
    n_estimators=500,
    max_depth=7,
    learning_rate=0.02,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8
)

# 训练并传入权重
model_advanced.fit(X_train_resampled, y_train_resampled, sample_weight=sample_weights)
# 预测与评估
y_pred_adv_encoded = model_advanced.predict(X_test_prep)
y_pred_adv = le.inverse_transform(y_pred_adv_encoded)
print("【SMOTE+XGBoost+权重优化】测试集准确率:", accuracy_score(y_test_encoded, y_pred_adv_encoded))
print("【SMOTE+XGBoost+权重优化】分类报告:\n", classification_report(y_test_encoded, y_pred_adv_encoded, target_names=le.classes_))

# 【极其重要】保存模型和对象
# 务必保存“工具对象”本身，千万不要保存 X_train_prep 等数据数组！
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
with open('cancer_model.pkl', 'wb') as f:
    pickle.dump(model_advanced, f)
print("\n✅ 恭喜！三个 pkl 文件（preprocessor.pkl, label_encoder.pkl, cancer_model.pkl）已成功正确保存！")

In [ ]:
import joblib
# 保存预处理器、模型和标签编码器
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(model_advanced, 'cancer_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("模型与预处理器保存成功！")

### 8.单样本预测验证

In [ ]:
# 1. 从测试集中抽取第1个样本（保持原始 DataFrame 格式）
sample = X_test.iloc[0:1]
# 2. 🌟 核心修复：使用训练好的 preprocessor 进行转换
sample_prep = preprocessor.transform(sample)
# 3. 使用模型进行预测，并通过 le 反转回中文
prediction_encoded = model_advanced.predict(sample_prep)
prediction = le.inverse_transform(prediction_encoded)
# 4. 打印结果
print("模型预测结果：", prediction)
print("该样本真实结果：", y_test.iloc[0])

In [ ]:
print("该样本的具体特征：")
print(sample.T)  # .T 表示转置，把 16 个特征竖着排成一列

9. 实验结论
本项目基于泛癌种临床与基因组数据，经过数据清洗、中文化映射、特征工程、模型深度调优及云端安全部署，成功构建了癌症样本类型（原发/转移/复发）的预测模型，得出以下核心结论：
### 1. 模型性能：在极度不平衡数据中实现技术攻坚
· 指标表现：经过 SMOTE 过采样与 XGBoost 样本权重双重强化后，模型在测试集上的整体准确率稳定在 0.865（约0.87），达到理想预期。在极端不平衡数据中，少数类取得突破性进展：“复发（Recurrent）”类别的 F1-score 从基线模型的极低水平提升至 0.19（受限于样本量仅35例，提升极为艰难）；“转移（Metastasis）”类别的 F1-score 稳定在 0.63；占绝大多数的“原发（Primary）”类别精确率与召回率均保持在 0.90 以上。
· 业务价值：通过引入 SMOTE生成合成样本 + 计算过采样后样本权重 的策略，模型不再盲目偏向多数类（原发灶），而是真正具备了识别高危复发/转移样本的能力，这对于临床辅助诊断具有极高的现实意义。
### 2. 业务洞察：警惕数据泄露，AI 不能替代业务常识
· 潜在风险：特征重要性分析显示，“肿瘤分期_IV期”排名第一。结合临床医学常识，IV期肿瘤通常意味着已经发生转移，这在机器学习中提示模型可能存在严重的“数据泄露（Data Leakage）”。模型可能“作弊”学到了“分期IV=转移”的捷径，而非真正的生物学标志物。
· 改进方向：未来在实际临床应用中，应剔除这类带有后验性质的特征，更多依赖“TMB（非同义）”、“肿瘤纯度”等独立分子生物标志物进行独立预测。AI 模型无法替代人工的病理诊断，只能作为辅助筛查的强力工具。
### 3. 闭环验证与工程化部署：密码学级别的模型安全
· 全流程闭环：从数据读取、中文化映射、特征工程、不平衡处理到模型训练，本实验完成了完整的机器学习数据挖掘闭环，成功构建了癌症样本类型预测系统。
· 安全架构（核心亮点）：针对医疗AI的数据隐私痛点，在将系统部署至 Streamlit Cloud 时，我摒弃了直接上传 .pkl 模型的传统做法，采用了“AES对称加密 + 密钥动态注入”的安全架构。在本地将模型加密为不可读的 .enc 密文上传至 GitHub，密钥仅存储于云端的 Streamlit Secrets 环境变量中。云端容器启动时在内存中解密，保证模型文件与患者数据绝对安全，完美兼顾了医疗 AI 应用的便捷性与数据隐私的绝对安全。
### 4. 数据安全底线
· 本次实践严格遵循数据隐私原则，所有数据处理、建模与调优均在地 Jupyter 环境本地闭环完成，绝不将包含患者敏感信息的原始 CSV 及未加密模型上传至任何外部 AI 平台。